# Regression intro: delivery distance and time

This notebook fits a straight line to delivery distance and minutes. You will read the slope, predict a value, measure R-squared, and check the residuals. The sklearn version is shown as a comment so the notebook runs with numpy and pandas only.

Put the CSV files for this dataset in a folder named `food_delivery` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("food_delivery/deliveries.csv")
d = raw.drop_duplicates(subset="delivery_id")
d = d[d["delivery_status"] == "Delivered"].dropna(subset=["distance_km", "actual_minutes"]).copy()
print("delivered rows used:", len(d))
d[["distance_km", "actual_minutes"]].describe().round(2)

## Step 1: Look at the scatter

Before a line, plot the points. Is the pattern roughly a straight band? Do you see a few far-off points?

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(d["distance_km"], d["actual_minutes"], s=4, alpha=0.3)
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Actual minutes")
ax.set_title("Delivery time against distance")
plt.tight_layout()
plt.show()

## Step 2: Fit the line

`np.polyfit(x, y, 1)` returns the slope first, then the intercept. The slope is the extra minutes for each extra km.

In [ ]:
x = d["distance_km"].to_numpy()
y = d["actual_minutes"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
print("slope (minutes per km):", round(slope, 2))
print("intercept (minutes at 0 km):", round(intercept, 2))

## Step 3: Predict

Use the line to predict minutes for a new distance. Stay inside the range of the data (about 0.6 to 14 km here).

In [ ]:
for km in [2, 5, 8]:
    print(f"{km} km -> predicted {round(intercept + slope * km, 1)} minutes")

## Step 4: Correlation and R-squared

With one x column, R-squared is the correlation squared. It is the share of the spread in minutes that the line explains.

In [ ]:
r = np.corrcoef(x, y)[0, 1]
print("correlation:", round(r, 3))
print("R-squared:", round(r ** 2, 3))

## Step 5: Residuals

A residual is actual minus predicted. Residuals should look like a random cloud around zero. A curve or a fan shape means the straight line is not a good model.

In [ ]:
pred = intercept + slope * x
resid = y - pred
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pred, resid, s=4, alpha=0.3)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Predicted minutes")
ax.set_ylabel("Residual (actual minus predicted)")
ax.set_title("Residual plot")
plt.tight_layout()
plt.show()
print("largest residual (minutes):", round(resid.max(), 1))

## Step 6: Effect of far trips

Refit without trips over 10 km. Compare the slopes. A big change means a few points are pulling the line.

In [ ]:
keep = d["distance_km"] <= 10
slope_trim, _ = np.polyfit(d.loc[keep, "distance_km"], d.loc[keep, "actual_minutes"], 1)
print("trips removed:", int((~keep).sum()))
print("slope with all rows:", round(slope, 2), "| slope without far trips:", round(slope_trim, 2))

## Step 7: The same model with scikit-learn (optional)

If scikit-learn is installed, the same fit takes a few lines. Uncomment them to try.

In [ ]:
# from sklearn.linear_model import LinearRegression
# model = LinearRegression().fit(d[["distance_km"]], d["actual_minutes"])
# print("slope:", round(model.coef_[0], 2), "intercept:", round(model.intercept_, 2))

## Findings template

- The slope says how many minutes each extra km adds. Write the number and its unit.
- R-squared says how much of the spread the line explains. Write what it does not explain (traffic, weather, the restaurant).
- Caveat: the line shows a link, not a cause. Do not claim that distance causes delay without a test.